# Model for the Streamlit app

A compact random forest on four inputs a user can type in (`age`, white blood cell count `wc`, hypertension `htn`,
diabetes `dm`). The notebook evaluates it with a split-first protocol, in which oversampling only touches the training
rows, then refits it on all 400 rows and saves it as `classifier.pkl` for `app.py`.

In [1]:
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

SEED = 7

In [2]:
df = pd.read_csv("kidney_disease.csv")[["age", "wc", "htn", "dm", "classification"]]
for col in ["htn", "dm", "classification"]:
    df[col] = df[col].str.strip()                        # ' yes', '\tno', 'ckd\t'
df["wc"] = pd.to_numeric(df["wc"], errors="coerce")       # '\t?' -> NaN
df["htn"] = df["htn"].map({"no": 0, "yes": 1})
df["dm"] = df["dm"].map({"no": 0, "yes": 1})
y = (df["classification"] == "ckd").astype(int).to_numpy()   # 1 = CKD, matching app.py
X = df[["age", "wc", "htn", "dm"]].to_numpy(dtype=float)     # column order matches app.py
print("missing values:", dict(zip(["age", "wc", "htn", "dm"], np.isnan(X).sum(axis=0))))
print("CKD:", y.sum(), "| not CKD:", (1 - y).sum())
df.head()

missing values: {'age': np.int64(9), 'wc': np.int64(106), 'htn': np.int64(2), 'dm': np.int64(2)}
CKD: 250 | not CKD: 150


,age,wc,htn,dm,classification
0,48.0,7800.0,1.0,1.0,ckd
1,7.0,6000.0,0.0,0.0,ckd
2,62.0,7500.0,0.0,1.0,ckd
3,48.0,6700.0,1.0,0.0,ckd
4,51.0,7300.0,0.0,0.0,ckd


In [3]:
def make_model():
    # Median imputation for every column (htn and dm are 0/1, so the median is the most common value),
    # oversampling of the minority class during fit only, then the random forest.
    return ImbPipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("oversample", RandomOverSampler(random_state=SEED)),
        ("model", RandomForestClassifier(max_depth=4, random_state=10)),
    ])


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
model = make_model().fit(X_train, y_train)
pred = model.predict(X_test)
print("held-out test accuracy: %.4f | ROC AUC: %.4f" % (accuracy_score(y_test, pred),
                                                        roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])))
print(classification_report(y_test, pred, target_names=["not CKD", "CKD"], digits=3))
print(confusion_matrix(y_test, pred))

held-out test accuracy: 0.8250 | ROC AUC: 0.9397
              precision    recall  f1-score   support

     not CKD      0.682     1.000     0.811        30
         CKD      1.000     0.720     0.837        50

    accuracy                          0.825        80
   macro avg      0.841     0.860     0.824        80
weighted avg      0.881     0.825     0.827        80

[[30  0]
 [14 36]]


In [4]:
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)
scores = cross_validate(make_model(), X, y, cv=cv, scoring=["accuracy", "f1", "roc_auc"], n_jobs=-1)
for k in ["accuracy", "f1", "roc_auc"]:
    s = scores[f"test_{k}"]
    print(f"repeated 5-fold CV {k:8s}: {s.mean():.4f} ± {s.std():.4f}")

repeated 5-fold CV accuracy: 0.8525 ± 0.0377
repeated 5-fold CV f1      : 0.8661 ± 0.0396
repeated 5-fold CV roc_auc : 0.9510 ± 0.0174


Four easy-to-enter inputs give up some accuracy compared with the full 24-feature models in the other notebooks.
After evaluation, the deployed model is refitted on all 400 rows.

In [5]:
final_model = make_model().fit(X, y)
with open("classifier.pkl", "wb") as f:
    pickle.dump(final_model, f)

# quick check with the same call app.py makes: [[age, wc, htn, dm]]
print(final_model.predict([[48, 7800, 1, 1], [30, 7000, 0, 0]]))

[1 0]


In [6]:
%%writefile app.py
import os
import pickle

import streamlit as st

# Path to the trained model produced by Model_Deployment.ipynb
MODEL_PATH = "classifier.pkl"


@st.cache_resource
def load_model(path):
    """Load the trained classifier once and cache it across reruns."""
    with open(path, "rb") as model_file:
        return pickle.load(model_file)


def predict(model, age, wc, htn, dm):
    """Return a human-readable CKD prediction for a single patient record."""
    # Pre-process the categorical user inputs into the encoding used at training time
    htn = 1 if htn == "yes" else 0
    dm = 1 if dm == "yes" else 0

    result = model.predict([[age, wc, htn, dm]])
    if result[0] == 0:
        return "Kidney Disease not detected"
    return "Kidney Disease found"


def main():
    """Define and render the Streamlit web page."""
    st.set_page_config(page_title="Kidney Disease Prediction", page_icon="🩺")

    # Front end header
    html_temp = """
    <div style="background-color:cyan;padding:13px">
    <h1 style="color:black;text-align:center;">Kidney Disease Prediction</h1>
    </div>
    """
    st.markdown(html_temp, unsafe_allow_html=True)
    st.write("")

    # Fail gracefully if the model has not been generated yet
    if not os.path.exists(MODEL_PATH):
        st.error(
            f"Trained model '{MODEL_PATH}' was not found. "
            "Please run Model_Deployment.ipynb first to generate it."
        )
        return

    model = load_model(MODEL_PATH)

    # Input widgets for the four features the model was trained on
    htn = st.selectbox("Hypertension (htn)", ("no", "yes"))
    dm = st.selectbox("Diabetes Mellitus (dm)", ("no", "yes"))
    age = st.number_input("Age", min_value=0, max_value=120, step=1)
    wc = st.number_input("White Blood Cell Count (wc)", min_value=0, step=100)

    # Run the prediction when the user clicks the button
    if st.button("Predict"):
        result = predict(model, age, wc, htn, dm)
        st.success(f"Report Result: {result}")


if __name__ == "__main__":
    main()

Overwriting app.py
